# Two-Scale on REAL Huginn - single-GPU (P100) run

Honest test of the "acceleration tracks reasoning depth" claim (David's deck number was computed on
np.random.randn, not the model). Single GPU: no multi-GPU threading, so the hook-contamination bug cannot occur.

Pipeline: debunk placeholder -> PARARULE depth 2-5 (shows depth==length confound) ->
LENGTH-MATCHED depth test (the real result) -> counting track/local cross-check -> early-exit -> figures -> zip.
Cached to /kaggle/working: re-run analysis cells without GPU. Rough time on P100: ~30-40 min first run.

In [1]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==4.53.3","accelerate","datasets"], check=False)
print("deps ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 92.4 MB/s eta 0:00:00
deps ready


In [2]:
import os, torch, numpy as np, pandas as pd
CACHE="/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(CACHE+"/results", exist_ok=True); os.makedirs(CACHE+"/figures", exist_ok=True)
def cpath(p): return os.path.join(CACHE,p)
NUM_STEPS=32; SEEDS=[0,1,2]; N_PER_DEPTH=10; POS_BINS=10
DEV="cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| gpus:", torch.cuda.device_count(), "| cache:", CACHE)

device: cuda | gpus: 2 | cache: /kaggle/working


In [3]:
# ---- Pappone two-scale metrics (single-pos = David's; all-pos vectorized, validated) ----
def accel_1pos(x):
    d=np.diff(x,axis=0)
    if len(d)<2: return np.array([])
    return np.linalg.norm(np.diff(d,axis=0),axis=1)
def orth_1pos(x):
    d=np.diff(x,axis=0)
    if len(d)<2: return np.array([])
    out=[]
    for i in range(len(d)-1):
        n1=np.linalg.norm(d[i]); n2=np.linalg.norm(d[i+1])
        out.append(float(np.dot(d[i],d[i+1])/(n1*n2)) if n1>0 and n2>0 else 0.0)
    return np.array(out)
def two_scale_allpos(traj):
    d=np.diff(traj,axis=0); sn=np.linalg.norm(d,axis=2)
    a=np.linalg.norm(np.diff(d,axis=0),axis=2)
    num=(d[1:]*d[:-1]).sum(axis=2); den=np.linalg.norm(d[1:],axis=2)*np.linalg.norm(d[:-1],axis=2)+1e-12
    cos=num/den
    return dict(accel=a, orth=cos, stepnorm=sn, mean_accel=a.mean(0), mean_orth=cos.mean(0))
def exit_step(a, pct=10.0):
    a=np.asarray(a)
    if a.size==0: return np.nan
    b=a<(pct/100.0)*a.max()
    for i in range(len(b)-1):
        if b[i] and b[i+1]: return int(i)
    return np.nan
def steps_to_settle(s, frac=0.1):
    s=np.asarray(s)
    if s.size==0: return np.nan
    idx=np.where(s<frac*s.max())[0]
    return int(idx[0]) if idx.size else int(len(s))
def norm_pos_bins(vals, nbins=POS_BINS):
    L=len(vals); out=np.full(nbins,np.nan)
    if L==0: return out
    idx=np.floor(np.linspace(0,nbins-1e-9,L)).astype(int)
    for b in range(nbins):
        mm=vals[idx==b]
        if mm.size: out[b]=np.nanmean(mm)
    return out
def row_metrics(m, L, meta, seed):
    return dict(**meta, seed=seed, seq_len=L,
        accel_answer=float(m["mean_accel"][-1]), orth_answer=float(m["mean_orth"][-1]),
        accel_content=float(np.nanmean(m["mean_accel"][:-1])), orth_content=float(np.nanmean(m["mean_orth"][:-1])),
        exit_answer=exit_step(m["accel"][:,-1]), settle_answer=steps_to_settle(m["stepnorm"][:,-1]))
def _rank(v): return pd.Series(np.asarray(v)).rank().values
def spearman(x,y): return float(np.corrcoef(_rank(x),_rank(y))[0,1])
CRIT={4:1.0,5:1.0,6:0.886,7:0.786,8:0.738,9:0.700,10:0.648,11:0.618,12:0.587}
def by_level(df,xc,yc):
    g=df.groupby(xc)[yc].mean(); r=spearman(g.index.values,g.values); n=len(g)
    c=CRIT.get(n, 1.96/np.sqrt(max(n-1,1)))
    return dict(rho=round(r,3), N=n, crit=round(float(c),3), sig=("sig" if abs(r)>c else "n.s."))
def partial_spearman(x,y,z):
    rx,ry,rz=_rank(x),_rank(y),_rank(z)
    def resid(a,b):
        B=np.c_[np.ones_like(b),b]; beta=np.linalg.lstsq(B,a,rcond=None)[0]; return a-B@beta
    return float(np.corrcoef(resid(rx,rz),resid(ry,rz))[0,1])
_t=np.random.randn(20,5,32); _v=two_scale_allpos(_t)
assert all(abs(_v["mean_accel"][p]-accel_1pos(_t[:,p,:]).mean())<1e-8 for p in range(5))
print("metrics OK")

metrics OK


In [4]:
# MODEL_LOAD (single GPU)
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL="tomg-group-umd/huginn-0125"; REV="bb6621b65e90b6a4b9b29ef88dc83866d450470c"
tok=AutoTokenizer.from_pretrained(MODEL, revision=REV)
model=AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, torch_dtype=torch.bfloat16, trust_remote_code=True).to(DEV).eval()
print("model loaded on", DEV)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

raven_config_minimal.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.74G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

model loaded on cuda


In [5]:
# EXTRACTION (single GPU, sequential - thread-safe by construction) + cache
def get_traj_allpos(prompt, num_steps=32, seed=0):
    mod=model.transformer.core_block[-1]; mod._forward_hooks.clear()
    lat=[]
    hk=mod.register_forward_hook(lambda mm,i,o: lat.append(o.detach().float().cpu()))
    try:
        torch.manual_seed(int(seed))
        ids=tok(prompt, return_tensors="pt").input_ids.to(DEV)
        with torch.no_grad(): model(input_ids=ids, num_steps=int(num_steps))
    finally:
        hk.remove()
    return torch.stack([l[0] for l in lat]).numpy(), int(ids.shape[1])

def extract_items(items, num_steps):
    out=[]
    for i,it in enumerate(items):
        traj,L=get_traj_allpos(it["prompt"], num_steps, it["seed"])
        m=two_scale_allpos(traj)
        out.append((row_metrics(m,L,it["meta"],it["seed"]), norm_pos_bins(m["mean_accel"])))
        del traj
        if (i+1)%25==0: print(f"   ...{i+1}/{len(items)}")
    return out

def cached_extract(csv, npy, items, num_steps):
    cp,np_=cpath(csv),cpath(npy)
    if os.path.exists(cp) and os.path.exists(np_):
        print("loaded cache", cp); return pd.read_csv(cp), np.load(np_)
    print(f"extracting {len(items)} items (single GPU)...")
    res=extract_items(items, num_steps)
    df=pd.DataFrame([r for r,_ in res]); mp=np.array([m for _,m in res])
    df.to_csv(cp,index=False); np.save(np_,mp); print("saved", cp)
    return df, mp

_a,_L=get_traj_allpos("Q: 1+1? A:", num_steps=8, seed=0); print("smoke:", _a.shape, "seq_len", _L)

smoke: (8, 9, 5280) seq_len 9


## 1. Debunk: reproduce David placeholder (np.random.randn, hand-coded 5+2*depth). No model.

In [6]:
np.random.seed(42); ph=[]
for i in range(40):
    depth=2+(i%4); traj=np.random.randn(32,5280)*0.1
    traj=traj*np.exp(-np.arange(32)/(5+depth*2))[:,None]
    ph.append(dict(depth=depth, mean_accel=accel_1pos(traj).mean(), mean_orth=orth_1pos(traj).mean()))
ph=pd.DataFrame(ph); ph.to_csv(cpath("results/two_scale_placeholder.csv"),index=False)
print(ph.groupby("depth")[["mean_accel","mean_orth"]].mean().round(3))
print("placeholder accel~depth rho =", round(spearman(ph.depth,ph.mean_accel),3), "(~1 by construction; no model)")

       mean_accel  mean_orth
depth                       
2           4.890     -0.497
3           5.841     -0.498
4           6.699     -0.498
5           7.451     -0.498
placeholder accel~depth rho = 0.969 (~1 by construction; no model)


## 2. PARARULE-Plus loader (real schema: meta.QDep)

In [7]:
from datasets import load_dataset
import ast, re
def _depth(ex):
    m=ex.get("meta")
    if isinstance(m,str):
        try: m=ast.literal_eval(m)
        except: m={}
    dp=(m or {}).get("QDep")
    if dp is None:
        mm=re.search(r'-D([2-5])-',str(ex.get("id",""))); dp=mm.group(1) if mm else None
    return int(dp) if dp is not None else None
def build_pararule(n_per_depth=N_PER_DEPTH):
    ds=load_dataset("qbao775/PARARULE-Plus","default",split="train",streaming=True).shuffle(seed=0,buffer_size=5000)
    need={2:n_per_depth,3:n_per_depth,4:n_per_depth,5:n_per_depth}; rows=[]
    for ex in ds:
        d=_depth(ex)
        if d is None or need.get(d,0)<=0: continue
        ctx=str(ex["context"]).strip(); q=str(ex["question"]).strip()
        rows.append(dict(depth=d, prompt=ctx+" Question: "+q+" Answer:")); need[d]-=1
        if all(v<=0 for v in need.values()): break
    return pd.DataFrame(rows)
pr=build_pararule(); print(pr.depth.value_counts().sort_index().to_dict()); print("ex:", pr.iloc[0].prompt[:160])

README.md: 0.00B [00:00, ?B/s]

'('Connection broken: IncompleteRead(0 bytes read, 15728640 more expected)', IncompleteRead(0 bytes read, 15728640 more expected))' thrown while requesting GET https://huggingface.co/datasets/qbao775/PARARULE-Plus/resolve/b69a66cba535e646cc3ace12b4eb672be78b44af/Depth2/PARARULE_Plus_Depth2_shuffled_train_huggingface.jsonl
Retrying in 1s [Retry 1/5].
HTTP Error 500 thrown while requesting GET https://huggingface.co/datasets/qbao775/PARARULE-Plus/resolve/b69a66cba535e646cc3ace12b4eb672be78b44af/Depth4/PARARULE_Plus_Depth4_shuffled_train_huggingface.jsonl
Retrying in 1s [Retry 1/5].
HTTP Error 500 thrown while requesting GET https://huggingface.co/datasets/qbao775/PARARULE-Plus/resolve/b69a66cba535e646cc3ace12b4eb672be78b44af/Depth3/PARARULE_Plus_Depth3_shuffled_train_huggingface.jsonl
Retrying in 1s [Retry 1/5].
'('Connection broken: IncompleteRead(0 bytes read, 15728640 more expected)', IncompleteRead(0 bytes read, 15728640 more expected))' thrown while requesting GET https://huggingfac

{2: 10, 3: 10, 4: 10, 5: 10}
ex: The dinosaur is dull. The dinosaur is slow. The bald eagle is fierce. The bald eagle is strong. The dinosaur likes the dog. The bald eagle visits the rabbit. Th


## 3. PARARULE depth 2-5, all positions (shows depth==length confound)

In [8]:
items=[{"prompt":r.prompt,"seed":sd,"meta":{"depth":int(r.depth)}} for _,r in pr.iterrows() for sd in SEEDS]
para, amap = cached_extract("results/pararule_depth.csv","results/pararule_depth_map.npy", items, NUM_STEPS)
print(para.groupby("depth")[["accel_answer","accel_content","seq_len"]].mean().round(3))

extracting 120 items (single GPU)...
   ...25/120
   ...50/120
   ...75/120
   ...100/120
saved /kaggle/working/results/pararule_depth.csv
       accel_answer  accel_content  seq_len
depth                                      
2            21.874         17.999    140.7
3            22.662         18.424    180.4
4            22.580         18.529    226.3
5            23.165         18.891    286.9


In [9]:
print("per-level Spearman + N:")
for col in ["accel_answer","accel_content","seq_len"]: print(f"  {col:14}", by_level(para,"depth",col))
print("partial | seq_len:")
for col in ["accel_answer","accel_content"]:
    print(f"  {col:14} raw={spearman(para.depth,para[col]):+.3f} partial|len={partial_spearman(para.depth,para[col],para.seq_len):+.3f}")
dep=para.depth.values
print("depth-corr by position bin (0=start..9=answer):", [round(spearman(dep,amap[:,b]),2) if np.isfinite(amap[:,b]).all() else None for b in range(POS_BINS)])
print("NOTE: seq_len rises with depth -> depth and length are collinear; partial is unreliable here (see cell 4).")

per-level Spearman + N:
  accel_answer   {'rho': 0.8, 'N': 4, 'crit': 1.0, 'sig': 'n.s.'}
  accel_content  {'rho': 1.0, 'N': 4, 'crit': 1.0, 'sig': 'n.s.'}
  seq_len        {'rho': 1.0, 'N': 4, 'crit': 1.0, 'sig': 'n.s.'}
partial | seq_len:
  accel_answer   raw=+0.327 partial|len=-0.047
  accel_content  raw=+0.360 partial|len=+0.618
depth-corr by position bin (0=start..9=answer): [0.71, 0.28, 0.33, 0.39, 0.39, 0.3, 0.23, 0.05, -0.06, 0.15]
NOTE: seq_len rises with depth -> depth and length are collinear; partial is unreliable here (see cell 4).


## 4. CENTERPIECE: length-matched depth test (stratified bands + within-band regression accel ~ depth + length)
Depth and length are collinear across the full range, but ADJACENT depths overlap. Within each overlap band we
hold length fixed and ask whether +1 reasoning step changes acceleration.

In [10]:
POOL=2000; SCAN=120000; K=60
pool={2:[],3:[],4:[],5:[]}; seen=0
ds=load_dataset("qbao775/PARARULE-Plus","default",split="train",streaming=True).shuffle(seed=5,buffer_size=30000)
for ex in ds:
    seen+=1
    if seen>SCAN: break
    d=_depth(ex)
    if d in pool and len(pool[d])<POOL:
        ctx=str(ex["context"]).strip(); q=str(ex["question"]).strip(); p=ctx+" Question: "+q+" Answer:"
        pool[d].append((len(tok(p).input_ids), p))
print("pool sizes:", {d:len(pool[d]) for d in [2,3,4,5]})
items=[]
for dlo,dhi in [(2,3),(3,4),(4,5)]:
    llo=np.array([x[0] for x in pool[dlo]]); lhi=np.array([x[0] for x in pool[dhi]])
    b0=max(llo.min(),lhi.min()); b1=min(llo.max(),lhi.max())
    lo_in=[p for L,p in pool[dlo] if b0<=L<=b1][:K]; hi_in=[p for L,p in pool[dhi] if b0<=L<=b1][:K]
    print(f"{dlo}-{dhi}: band [{int(b0)},{int(b1)}] n_lo={len(lo_in)} n_hi={len(hi_in)}")
    for p in lo_in: items.append({"prompt":p,"seed":0,"meta":{"adj":f"{dlo}-{dhi}","hi":0}})
    for p in hi_in: items.append({"prompt":p,"seed":0,"meta":{"adj":f"{dlo}-{dhi}","hi":1}})
band,_=cached_extract("results/band.csv","results/band_map.npy", items, NUM_STEPS)
print("\n=== +1 depth within a common length band ===")
for adj in ['2-3','3-4','4-5']:
    s=band[band.adj==adj]
    if s.hi.nunique()<2: print(f"  {adj}: n/a"); continue
    lo=s[s.hi==0]; hi=s[s.hi==1]
    for metric in ["accel_content","accel_answer"]:
        X=np.c_[np.ones(len(s)), s.hi.values, s.seq_len.values]; beta=np.linalg.lstsq(X,s[metric].values,rcond=None)[0]
        print(f"  {adj} {metric}: mean_lo={lo[metric].mean():.2f} mean_hi={hi[metric].mean():.2f} | len_lo={lo.seq_len.mean():.0f} len_hi={hi.seq_len.mean():.0f} | depth-coef|len={beta[1]:+.3f}")

pool sizes: {2: 2000, 3: 2000, 4: 2000, 5: 2000}
2-3: band [149,187] n_lo=60 n_hi=60
3-4: band [185,233] n_lo=60 n_hi=60
4-5: band [220,282] n_lo=60 n_hi=60
extracting 360 items (single GPU)...
   ...25/360
   ...50/360
   ...75/360
   ...100/360
   ...125/360
   ...150/360
   ...175/360
   ...200/360
   ...225/360
   ...250/360
   ...275/360
   ...300/360
   ...325/360
   ...350/360
saved /kaggle/working/results/band.csv

=== +1 depth within a common length band ===
  2-3 accel_content: mean_lo=17.81 mean_hi=18.87 | len_lo=171 len_hi=153 | depth-coef|len=+1.565
  2-3 accel_answer: mean_lo=22.62 mean_hi=22.13 | len_lo=171 len_hi=153 | depth-coef|len=+0.203
  3-4 accel_content: mean_lo=18.25 mean_hi=19.00 | len_lo=210 len_hi=189 | depth-coef|len=+1.266
  3-4 accel_answer: mean_lo=22.86 mean_hi=22.10 | len_lo=210 len_hi=189 | depth-coef|len=-0.650
  4-5 accel_content: mean_lo=18.40 mean_hi=19.18 | len_lo=253 len_hi=224 | depth-coef|len=+1.184
  4-5 accel_answer: mean_lo=23.83 mean_hi=22.

## 5. Counting track/local - length-matched cross-check (different reasoning type)

In [11]:
def counting_prompt(n_ops, kind, seed=0):
    import random; rng=random.Random(seed*997+n_ops); ops=[rng.choice([1,-1]) for _ in range(n_ops)]
    body="Start at 0. "+" ".join(("add 1" if o>0 else "subtract 1") for o in ops)+"."
    if kind=="track": return body+" Question: what is the final total? Answer:"
    return body+" Question: what was the last instruction? Answer:"
citems=[{"prompt":counting_prompt(n,k,sd),"seed":sd,"meta":{"n_ops":n,"kind":k}} for n in [4,8,16,24,32,48] for k in ["track","local"] for sd in [0,1,2]]
cnt,_=cached_extract("results/counting.csv","results/counting_map.npy", citems, 64)
for k in ["track","local"]: print(f"  {k:6}", by_level(cnt[cnt.kind==k],"n_ops","accel_answer"))
piv=cnt.groupby(["n_ops","kind"])["accel_answer"].mean().unstack(); piv["track-local"]=piv["track"]-piv["local"]; print(piv.round(2))

extracting 36 items (single GPU)...
   ...25/36
saved /kaggle/working/results/counting.csv
  track  {'rho': -0.714, 'N': 6, 'crit': 0.886, 'sig': 'n.s.'}
  local  {'rho': -0.429, 'N': 6, 'crit': 0.886, 'sig': 'n.s.'}
kind   local  track  track-local
n_ops                           
4      10.95  11.75         0.80
8      11.33  11.52         0.19
16     11.39  11.87         0.49
24     11.54  11.54        -0.00
32     10.94  11.34         0.40
48     10.78  11.33         0.55


## 6. Early-exit (Pappone) on real trajectories

In [12]:
print(para[["exit_answer","settle_answer"]].describe().round(2))
v=para.dropna(subset=["exit_answer"])
if len(v): print("corr(exit,settle)=", round(spearman(v.exit_answer,v.settle_answer),3), "| fires on", round(100*len(v)/len(para)),"%")

       exit_answer  settle_answer
count       120.00         120.00
mean         14.28          19.63
std           1.00           3.46
min          12.00          14.00
25%          14.00          16.00
50%          14.00          20.00
75%          15.00          22.00
max          17.00          27.00
corr(exit,settle)= 0.434 | fires on 100 %


In [13]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(15,4)); dep=para.depth.values
pr_=[spearman(dep,amap[:,b]) if np.isfinite(amap[:,b]).all() else np.nan for b in range(POS_BINS)]
ax[0].plot(range(POS_BINS),pr_,"o-"); ax[0].axhline(0,color="gray",lw=.7); ax[0].set_title("depth-corr by position"); ax[0].set_xlabel("bin 0=start..9=answer")
rd=para.groupby("depth")["accel_content"].mean(); ax[1].plot(rd.index,rd.values,"o-",label="REAL"); pp=ph.groupby("depth")["mean_accel"].mean(); ax[1].plot(pp.index,pp.values,"s--",label="placeholder"); ax[1].set_title("accel vs depth"); ax[1].legend()
ro=para.groupby("depth")["orth_answer"].mean(); ax[2].plot(ro.index,ro.values,"o-",label="REAL"); ax[2].axhline(-0.5,color="red",ls="--",label="noise -0.5"); ax[2].set_title("orthogonality vs depth"); ax[2].legend()
plt.tight_layout(); plt.savefig(cpath("figures/two_scale_real.png"),dpi=130); print("saved figure")

saved figure


In [14]:
# ---- collect everything + honest FINDINGS into a zip ----
import glob, zipfile, time
L=["TWO-SCALE ON REAL HUGINN (single-GPU) - honest summary","="*52,"",
   "1. David deck numbers (accel rho=0.997, orth=-0.5) = np.random.randn + hand-coded 5+2*depth. No model.",""]
try:
    L+=["2. PARARULE depth 2-5: accel nearly flat; seq_len rises with depth -> depth and length collinear,",
        "   cannot be separated on full range (partial unreliable).",
        "   accel_content by depth: "+", ".join(f"{v:.2f}" for v in para.groupby('depth')['accel_content'].mean()),""]
except Exception: pass
try:
    L+=["3. LENGTH-MATCHED (adjacent depths, length fixed): depth-coef|len per band:"]
    for adj in ['2-3','3-4','4-5']:
        s=band[band.adj==adj]
        if s.hi.nunique()<2: continue
        X=np.c_[np.ones(len(s)),s.hi.values,s.seq_len.values]; b=np.linalg.lstsq(X,s['accel_content'].values,rcond=None)[0]
        L+=[f"   {adj}: accel_content depth-coef|len = {b[1]:+.3f} (n={len(s)})"]
    L+=[""]
except Exception: pass
L+=["VERDICT:",
    " - David's evidence was a random-noise artifact (certain).",
    " - Answer-token acceleration does not track depth.",
    " - At matched length, +1 depth raises CONTENT acceleration on low-mid steps (real signal beyond length),",
    "   but 'depth' co-varies with logical rule-count; high depths under-sampled. Promising, not conclusive."]
open(cpath("results/FINDINGS.txt"),"w").write("\n".join(L)); print("\n".join(L))
stamp=time.strftime("%Y%m%d_%H%M"); z=cpath(f"two_scale_all_{stamp}.zip")
files=[f for p in ["results/*.csv","results/*.npy","results/*.txt","figures/*.png"] for f in glob.glob(cpath(p))]
with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zz:
    for f in files: zz.write(f, os.path.relpath(f,CACHE))
print(f"\nZIPPED {len(files)} files -> {z} (download from Output tab)")

TWO-SCALE ON REAL HUGINN (single-GPU) - honest summary

1. David deck numbers (accel rho=0.997, orth=-0.5) = np.random.randn + hand-coded 5+2*depth. No model.

2. PARARULE depth 2-5: accel nearly flat; seq_len rises with depth -> depth and length collinear,
   cannot be separated on full range (partial unreliable).
   accel_content by depth: 18.00, 18.42, 18.53, 18.89

3. LENGTH-MATCHED (adjacent depths, length fixed): depth-coef|len per band:
   2-3: accel_content depth-coef|len = +1.565 (n=120)
   3-4: accel_content depth-coef|len = +1.266 (n=120)
   4-5: accel_content depth-coef|len = +1.184 (n=120)

VERDICT:
 - David's evidence was a random-noise artifact (certain).
 - Answer-token acceleration does not track depth.
 - At matched length, +1 depth raises CONTENT acceleration on low-mid steps (real signal beyond length),
   but 'depth' co-varies with logical rule-count; high depths under-sampled. Promising, not conclusive.

ZIPPED 9 files -> /kaggle/working/two_scale_all_20260713_223

## Summary
Fill from the run. Key cells: 3 (PARARULE depth, confound), 4 (length-matched depth-coef|len = the real test), 5 (counting).
If depth-coef|len is clearly >0 across 2-3/3-4 with len_lo~len_hi -> acceleration tracks depth beyond length on low-mid steps.